In [4]:
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd

# 0. Definición de rutas adaptables al entorno (cuadernos en 'notebooks' o raíz)
BASE_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_DIR = BASE_DIR / 'data'

drivers = pd.read_csv(DATA_DIR / 'drivers.csv')
timesheet = pd.read_csv(DATA_DIR / 'timesheet.csv')

# 1. Promedio de hours-logged y miles-logged por conductor (sin semana)
avg_timesheet = (
    timesheet.groupby('driverId')[['hours-logged', 'miles-logged']]
    .mean()
    .reset_index()
)

# 2. Tabla timesheet_with_means (agrega mean_hours-logged conservando los registros originales)
timesheet_with_means = timesheet.copy()
timesheet_with_means['mean_hours-logged'] = timesheet_with_means.groupby(
    'driverId'
)['hours-logged'].transform('mean')

# 3. Tabla timesheet_below (registros donde hours-logged < mean_hours-logged)
timesheet_below = timesheet_with_means[
    timesheet_with_means['hours-logged']
    < timesheet_with_means['mean_hours-logged']
].copy()

# 4. Tabla sum_timesheet (suma acumulada de horas y millas por conductor)
sum_timesheet = (
    timesheet.groupby('driverId')[['hours-logged', 'miles-logged']]
    .sum()
    .reset_index()
)

# 5. Tabla min_max_timesheet (mínimo y máximo de horas por conductor)
min_max_timesheet = (
    timesheet.groupby('driverId')['hours-logged']
    .agg(min_hours_logged='min', max_hours_logged='max')
    .reset_index()
)

# 6. Tabla summary (combinando sum_timesheet con driverId y name de drivers)
summary = sum_timesheet.merge(
    drivers[['driverId', 'name']], on='driverId', how='inner'
)
summary = summary[['driverId', 'hours-logged', 'miles-logged', 'name']]

# 7. Guardar summary.csv (sin índice)
summary.to_csv(BASE_DIR / 'summary.csv', index=False)

# 8. Tabla top10 (los diez conductores con más millas, ordenados de mayor a menor)
top10 = summary.sort_values(by='miles-logged', ascending=False).head(10).copy()

# 9. Gráfico de barras horizontales (top10_drivers.png)
top10_plot = top10.sort_values(by='miles-logged', ascending=True)

plt.figure(figsize=(10, 6))
bars = plt.barh(
    top10_plot['name'],
    top10_plot['miles-logged'],
    color='#2b5c8f',
    edgecolor='none',
)

plt.xlabel('Cantidad de Millas Registradas (miles-logged)', fontsize=11)
plt.ylabel('Nombre del Conductor', fontsize=11)
plt.title(
    'Top 10 Conductores con Mayor Millaje Registrado',
    fontsize=13,
    fontweight='bold',
    pad=15,
)
plt.grid(axis='x', linestyle='--', alpha=0.5)

# Etiquetas numéricas en las barras
for bar in bars:
  width = bar.get_width()
  plt.text(
      width + 200,
      bar.get_y() + bar.get_height() / 2,
      f'{width:,.0f}',
      va='center',
      ha='left',
      fontsize=9,
      fontweight='bold',
      color='#333333',
  )

plt.xlim(0, max(top10_plot['miles-logged']) * 1.12)
plt.tight_layout()
plt.savefig(BASE_DIR / 'top10_drivers.png', dpi=300)
plt.close()